# LangGraph: Генерация учебных материалов по математической статистике
## с использованием локальной модели Qwen2.5-1.5B-Instruct (float16)

## 1. Установка зависимостей

In [ ]:
!pip install -q langchain langgraph langchain_huggingface transformers torch accelerate pydantic jinja2

## 2. Импорты и настройка окружения

In [17]:
import os
import re
import operator
from typing import Annotated, List, Any
from pydantic import BaseModel, Field
from jinja2 import Template

# LangChain и LangGraph
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.constants import Send
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

# HuggingFace
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

C:\Users\Admin\AppData\Local\Temp\ipykernel_28824\355241950.py:11: LangGraphDeprecatedSinceV10: Importing Send from langgraph.constants is deprecated. Please use 'from langgraph.types import Send' instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  from langgraph.constants import Send


## 3. Загрузка локальной модели Qwen2.5-1.5B-Instruct (float16)

In [18]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16

print(f"Загрузка модели {MODEL_NAME} в формате {DTYPE} на устройство {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    max_length=None,
    temperature=0.7,
    do_sample=True,
    top_p=0.95,
    repetition_penalty=1.1
)

llm = HuggingFacePipeline(pipeline=pipe)
print("Модель загружена!")

Загрузка модели Qwen/Qwen2.5-1.5B-Instruct в формате torch.float16 на устройство cpu...


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 610.30it/s]


Модель загружена!


## 4. Функция извлечения вопросов из текста (без JSON)

In [19]:
def extract_questions_from_text(text: str) -> List[str]:
    """
    Извлекает нумерованные вопросы, отбрасывая строки, похожие на инструкции.
    """
    lines = text.split('\n')
    questions = []
    
    # Регулярное выражение для нумерованных пунктов
    pattern = re.compile(r'^\s*(?:\d+[\.\)])\s+(.+)$')
    
    # Слова-маркеры, указывающие на настоящий вопрос
    question_markers = ['как', 'что', 'почему', 'зачем', 'когда', 'где', 'какой', 'какая',
                        'какие', 'какое', 'сколько', 'можно ли', 'в чём', 'объясните',
                        'опишите', 'сравните', 'поясните', 'верно ли', 'определите']
    
    # Шумные слова, которые не должны быть в вопросе (обычно это требования из промпта)
    noise_words = ['полнота', 'ясность', 'выкладки', 'пошаговыми', 'выводами',
                   'математические', 'формулы', 'объясните каждый символ', 'логический шаг',
                   'требования', 'примеры и связь', 'ответ на русском языке']
    
    for line in lines:
        line = line.strip()
        match = pattern.match(line)
        if not match:
            continue
        
        q_text = match.group(1).strip()
        
        # Проверка 1: должен быть вопросительный знак
        if '?' not in q_text and '?' not in q_text:
            continue
        
        # Проверка 2: не слишком короткий
        if len(q_text) < 15:
            continue
        
        # Проверка 3: нет шумных слов
        lower_q = q_text.lower()
        if any(noise in lower_q for noise in noise_words):
            continue
        
        # Проверка 4: хотя бы одно вопросительное слово или начинается с глагола в повелительном наклонении (объясните, опишите)
        has_marker = any(marker in lower_q for marker in question_markers)
        if not has_marker and not (lower_q.startswith('объясните') or lower_q.startswith('опишите')):
            continue
        
        questions.append(q_text)
    
    # Если ничего не нашли – fallback с общим вопросом
    if not questions:
        questions = ["Сформулируйте вопрос более конкретно, используя математические термины"]
    
    return questions[:5]  # максимум 5 вопросов

## 5. Промпты (на русском языке) для математической статистики

In [20]:
from jinja2 import Template

GENERATING_CONTENT_PROMPT = Template("""
Ты — профессор математической статистики. Подготовь исчерпывающий учебный материал по вопросу:
{{ exam_question }}

Требования:
1. Полнота, ясность, математические выкладки с пошаговыми выводами.
2. Примеры и связь с реальными статистическими методами.
3. Ответ на русском языке.
4. Математика: Если для полного понимания требуются математические формулы, концепции или производные, представьте каждую формулу с подробным, пошаговым выводом.
 - Объясните каждый символ и каждый логический шаг.
 - Для встроенных математических формул используйте одиночные знаки доллара:
 Пример: Скорость света: `$c = 3 \times 10^8\ м/с$`
 - Для отображения математических формул (блоков) используйте двойные знаки доллара. Например:
        $$
        \vec{F} = m \vec{a} = m \frac{d\vec{v}}{dt}
        $$                                

Начни сразу с материала, без лишних слов.
""")

GEN_QUESTION_PROMPT = Template("""
Ты — преподаватель математической статистики.
Проанализируй учебный материал по теме «{{ exam_question }}».

Материал:
{{ study_material }}

Сформулируй 5 дополнительных вопросов, которые в этом материале НЕ раскрыты или раскрыты слишком поверхностно.
Каждый вопрос должен:
- быть на русском языке,
- относиться к теме,
- требовать более глубокого понимания,
- заканчиваться вопросительным знаком,
- начинаться с номера и точки (например, «1. Как ...?»).

Не добавляй никаких пояснений, заголовков или лишнего текста. Выведи только 5 пронумерованных вопросов.

Пример правильного ответа:
1. Как выводится формула стандартной ошибки разности средних?
2. Почему для критерия Стьюдента важно нормальное распределение?
3. Что произойдёт с мощностью критерия при увеличении выборки?
4. Как проверить предположение о равенстве дисперсий перед t-тестом?
5. В каких случаях используют одностороннюю альтернативу?

Теперь напиши 5 своих вопросов по указанному материалу.
""")

REFINE_QUESTION_PROMPT = Template("""
Ты — преподаватель математической статистики. Уточни вопросы на основе обратной связи пользователя.

Экзаменационный вопрос: {{ exam_question }}
Учебный материал: {{ study_material }}
Текущие вопросы: {{ current_questions }}
Обратная связь пользователя: {{ user_feedback }}

Если пользователь выразил удовлетворение (слова: "дальше", "хорошо", "отлично", "всё ок", "давай", "подходит"), то просто напиши слово "FINALIZE".
Иначе предоставь исправленный список вопросов, каждый с новой строки, начиная с номера и точки или скобки.
""")

GEN_ANSWER_PROMPT = Template("""
Ты — отличник по математической статистике. Дай развёрнутый ответ на вопрос:
{{ exam_question }}

Ответ должен содержать:
- Чёткие определения и формулировки.
- Математические формулы с пояснением символов.
- Примеры применения в статистике.
- Для встроенных математических формул используйте одинарные знаки доллара:
 Пример: Скорость света: `$c = 3 умножить на 10^8 м/с`
- Для отображения (блока) математических формул используйте двойные знаки доллара:
 Пример:
 $$
\vec{F} = m \vec{a} = m \frac{d\vec{v}}{dt}
$$
- Ответ на русском языке.
""")

<>:14: SyntaxWarning: invalid escape sequence '\ '
<>:14: SyntaxWarning: invalid escape sequence '\ '
C:\Users\Admin\AppData\Local\Temp\ipykernel_28824\3163792007.py:14: SyntaxWarning: invalid escape sequence '\ '
  Пример: Скорость света: `$c = 3 \times 10^8\ м/с$`


## 6. Модели данных (Pydantic) для состояния графа

In [21]:
class ExamState(BaseModel):
    exam_question: str = Field(default="", description="Экзаменационный вопрос")
    generated_material: str = Field(default="", description="Сгенерированный учебный материал")
    gap_questions: List[str] = Field(default_factory=list, description="Дополнительные вопросы")
    gap_q_n_a: Annotated[List[str], operator.add] = Field(default_factory=list, description="Вопросы и ответы")
    feedback_messages: List[Any] = Field(default_factory=list, description="История обратной связи HITL")

## 7. Утилиты

In [22]:
from IPython.display import Image as IPythonImage, display, clear_output

def display_graph(app, xray=0):
    display(IPythonImage(app.get_graph(xray=xray).draw_mermaid_png()))

def save_to_markdown(state):
    md = []
    md.append(f"# Экзаменационный вопрос\n\n{state['exam_question']}\n")
    if state.get('generated_material'):
        md.append("## Основной материал\n")
        md.append(state['generated_material'])
        md.append("")
    if state.get('gap_q_n_a'):
        md.append("## Дополнительные вопросы и ответы")
        for i, qna in enumerate(state['gap_q_n_a'], 1):
            md.append(f"{i}. {qna}")
        md.append("")
    return "\n".join(md)

## 8. Построение графа LangGraph

In [23]:
async def generating_content_node(state: ExamState) -> Command:
    """Генерация основного материала"""
    prompt = GENERATING_CONTENT_PROMPT.render(exam_question=state.exam_question)
    response = await llm.ainvoke([SystemMessage(content=prompt)])
    # Очищаем вывод перед показом нового материала
    clear_output(wait=True)
    print("=== СГЕНЕРИРОВАННЫЙ МАТЕРИАЛ ===")
    print(response)
    print("================================\n")
    return Command(goto="generating_questions", update={"generated_material": response})

async def generating_questions_node(state: ExamState) -> Command:
    """Генерация/уточнение дополнительных вопросов"""
    # Первый запуск – генерация начальных вопросов
    if not state.feedback_messages:
        prompt = GEN_QUESTION_PROMPT.render(
            exam_question=state.exam_question,
            study_material=state.generated_material
        )
        response = await llm.ainvoke([SystemMessage(content=prompt)])
        questions = extract_questions_from_text(response)
        # Показываем вопросы пользователю
        print("=== ПРЕДЛОЖЕННЫЕ ДОПОЛНИТЕЛЬНЫЕ ВОПРОСЫ ===")
        for i, q in enumerate(questions, 1):
            print(f"{i}. {q}")
        print("===========================================\n")
        
        initial_msg = AIMessage(content="\n".join([f"{i+1}. {q}" for i, q in enumerate(questions)]))
        return Command(
            goto="generating_questions",
            update={
                "gap_questions": questions,
                "feedback_messages": [initial_msg]
            }
        )

    # HITL: запрос обратной связи
    print("Ожидание обратной связи по вопросам...")
    user_feedback = interrupt("Введите ваш ответ (можно написать 'дальше', если всё хорошо, или дать правки): ")

    # Анализируем ответ пользователя
    approval_keywords = ["дальше", "хорошо", "отлично", "всё ок", "давай", "подходит", "ok", "yes", "go", "finalize"]
    if any(keyword in user_feedback.lower() for keyword in approval_keywords):
        # Переходим к ответам
        sends = [Send("answer_question", {"question": q}) for q in state.gap_questions]
        return Command(
            goto=sends,
            update={
                "feedback_messages": []
            }
        )
    else:
        # Уточняем вопросы с учётом обратной связи
        refine_prompt = REFINE_QUESTION_PROMPT.render(
            exam_question=state.exam_question,
            study_material=state.generated_material,
            current_questions=state.gap_questions,
            user_feedback=user_feedback
        )
        response = await llm.ainvoke([SystemMessage(content=refine_prompt)])
        if "FINALIZE" in response.strip().upper():
            sends = [Send("answer_question", {"question": q}) for q in state.gap_questions]
            return Command(
                goto=sends,
                update={"feedback_messages": []}
            )
        else:
            new_questions = extract_questions_from_text(response)
            print("=== УТОЧНЁННЫЕ ВОПРОСЫ ===")
            for i, q in enumerate(new_questions, 1):
                print(f"{i}. {q}")
            print("==========================\n")
            refined_msg = AIMessage(content="\n".join([f"{i+1}. {q}" for i, q in enumerate(new_questions)]))
            return Command(
                goto="generating_questions",
                update={
                    "gap_questions": new_questions,
                    "feedback_messages": state.feedback_messages + [refined_msg]
                }
            )

async def answer_question_node(data: dict) -> Command:
    """Генерация ответа на один вопрос (параллельно)"""
    prompt = GEN_ANSWER_PROMPT.render(exam_question=data['question'])
    response = await llm.ainvoke([SystemMessage(content=prompt)])
    return Command(
        goto=END,
        update={"gap_q_n_a": [f"##{data['question']}\n\n{response}"]}
    )

# Сборка графа
workflow = StateGraph(ExamState)
workflow.add_node("generating_content", generating_content_node)
workflow.add_node("generating_questions", generating_questions_node)
workflow.add_node("answer_question", answer_question_node)
workflow.add_edge(START, "generating_content")

checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)

print("Граф успешно создан.")

Граф успешно создан.


## 9. Запуск примера

In [ ]:
import asyncio

async def run_example():
    thread_config = {"configurable": {"thread_id": "stats_example"}}
    exam_question = "Раскройте тему: 'Параметрический критерий различий и сдвигов: т - Критерий Стьюдента.'"

    input_state = {"exam_question": exam_question}

    while True:
        async for event in app.astream(input_state, thread_config, stream_mode="updates"):
            # просто прогоняем события, вывод уже сделан внутри узлов
            pass

        state_snapshot = await app.aget_state(thread_config)
        if state_snapshot.interrupts:
            user_input = input()  # прерывание уже вывело запрос, просто ждём ввода
            input_state = Command(resume=user_input)
        else:
            break

    final_state = state_snapshot.values
    md_text = save_to_markdown(final_state)
    with open("statistics_lecture.md", "w", encoding="utf-8") as f:
        f.write(md_text)
    print("✅ Готово! Результат сохранён в statistics_lecture.md")


In [25]:
await run_example()

=== СГЕНЕРИРОВАННЫЙ МАТЕРИАЛ ===
System: 
Ты — профессор математической статистики. Подготовь исчерпывающий учебный материал по вопросу:
Раскройте тему: 'Параметрический критерий различий и сдвигов: т - Критерий Стьюдента.'

Требования:
1. Полнота, ясность, математические выкладки с пошаговыми выводами.
2. Примеры и связь с реальными статистическими методами.
3. Ответ на русском языке.
4. Математика: Если для полного понимания требуются математические формулы, концепции или производные, представьте каждую формулу с подробным, пошаговым выводом.
 - Объясните каждый символ и каждый логический шаг.
 - Для встроенных математических формул используйте одиночные знаки доллара:
 Пример: Скорость света: `$c = 3 	imes 10^8\ м/с$`
 - Для отображения математических формул (блоков) используйте двойные знаки доллара. Например:
        $$
        ec{F} = m ec{a} = m rac{dec{v}}{dt}
        $$                                

Начни сразу с материала, без лишних слов. Не забудь о важности использо